In [ ]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 20.4 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import torch
import re
import ast
import black
import autopep8
import requests
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from transformers import pipeline

In [ ]:
class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95

device = 0 if torch.cuda.is_available() else -1

print("Loading Granite model...")

model_pipe = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME,
    device=device
)

print("Model Loaded Successfully")

In [ ]:
def detect_python_code(text):
    patterns = [
        r'def\s+\w+\s*\(',
        r'class\s+\w+',
        r'import\s+\w+',
        r'from\s+\w+\s+import',
        r'if\s+.*:',
        r'for\s+.*\s+in\s+',
        r'while\s+.*:'
    ]

    for pattern in patterns:
        if re.search(pattern, text):
            return True
    return False

In [ ]:
def check_syntax(code):
    try:
        ast.parse(code)
        return True, None
    except SyntaxError as e:
        return False, f"Syntax Error (line {e.lineno}): {e.msg}"

In [ ]:
def fix_code_indentation(code):
    lines = code.split("\n")
    fixed = []

    for line in lines:
        line = line.rstrip()
        line = line.replace("\t", "    ")
        fixed.append(line)

    return "\n".join(fixed)

In [ ]:
def fix_code_brackets(code):
    brackets = {"(": ")", "[": "]", "{": "}"}
    stack = []

    for char in code:
        if char in brackets:
            stack.append(brackets[char])
        elif char in brackets.values():
            if stack and char == stack[-1]:
                stack.pop()

    while stack:
        code += stack.pop()

    return code

In [ ]:
def format_code(code):
    try:
        return black.format_str(code, mode=black.FileMode())
    except:
        return autopep8.fix_code(code)

In [ ]:
def fix_python_code(code):
    code = fix_code_indentation(code)
    code = fix_code_brackets(code)

    valid, error = check_syntax(code)

    if not valid:
        return f"❌ {error}\n\nFixed Code Attempt:\n{code}"

    code = format_code(code)

    return f"✅ Fixed Python Code:\n\n{code}"

In [ ]:
conversation_history = []

In [ ]:
def generate_response(user_message, temperature, max_tokens, top_p):

    if detect_python_code(user_message):
        return fix_python_code(user_message)

    conversation_history.append({"role": "user", "content": user_message})

    response = model_pipe(
        user_message,
        max_new_tokens=int(max_tokens),
        temperature=temperature,
        top_p=top_p
    )

    answer = response[0]["generated_text"]

    conversation_history.append({"role": "assistant", "content": answer})

    return answer

In [ ]:
def download_txt():
    text = ""

    for msg in conversation_history:
        text += f"{msg['role'].upper()} : {msg['content']}\n\n"

    with open("chat_history.txt", "w") as f:
        f.write(text)

    return "chat_history.txt"

In [ ]:
def download_pdf():

    styles = getSampleStyleSheet()
    story = []

    for msg in conversation_history:
        text = f"<b>{msg['role'].upper()}</b>: {msg['content']}"
        story.append(Paragraph(text, styles['Normal']))
        story.append(Spacer(1, 10))

    file = "chat_history.pdf"

    doc = SimpleDocTemplate(file)
    doc.build(story)

    return file

In [ ]:
def chat_interface(message, temperature, max_tokens, top_p):
    reply = generate_response(message, temperature, max_tokens, top_p)
    return reply


interface = gr.Interface(
    fn=chat_interface,
    inputs=[
        gr.Textbox(lines=3, label="Enter Message or Python Code"),
        gr.Slider(0.1, 1.0, value=0.7, label="Temperature"),
        gr.Slider(50, 1024, value=256, step=50, label="Max Tokens"),
        gr.Slider(0.1, 1.0, value=0.95, label="Top P")
    ],
    outputs="text",
    title="The Syntax Surgeon - AI Code Fixer"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bafd47a94ab4c3e3f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
